In [1]:
# Consolidated imports
import os
import glob
from pathlib import Path
import json
from datetime import datetime
import warnings

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# SciPy signal utilities
from scipy.signal import spectrogram, butter, filtfilt, resample_poly
from scipy import signal

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Notebook utilities
from IPython.display import Audio, display

warnings.filterwarnings('ignore')
print('Imports loaded')
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Imports loaded
PyTorch version: 2.5.1+cu121
CUDA available: False


In [12]:
# === CONFIGURATION ===
# Point to the model run directory you want to evaluate
MODEL_RUN_DIR = '/fs/scratch/<allocation>/model_runs/powerline_wavelet_cnn_20251117_211420'  # Update with your v4 run timestamp

# Load configuration from the training run
config_path = os.path.join(MODEL_RUN_DIR, 'config.json')
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        CONFIG = json.load(f)
    print(f"✓ Loaded config from: {config_path}")
else:
    print(f"❌ Config file not found: {config_path}")
    # Fallback configuration
    CONFIG = {
        'DATA_FOLDERS': ['/fs/scratch/<allocation>/May29_Alice'],
        'SAMPLE_RATE': 200_000,
        'CHUNK_DURATION': 0.1,
        'CARRIER_FREQ': 20000,
        'BANDWIDTH': 10000,
        'USB_CUTOFF_HZ': 20000,
        'DEVICE': 'cuda' if torch.cuda.is_available() else 'cpu',
        'TRAIN_SPLIT': 0.8,
        'OUTPUT_DIR': MODEL_RUN_DIR
    }

print("\nConfiguration:")
for key, value in CONFIG.items():
    if isinstance(value, list) and len(value) > 2:
        print(f"  {key}: [{len(value)} items]")
    else:
        print(f"  {key}: {value}")


# Create evaluation output directoryos.makedirs(eval_output_dir, exist_ok=True)
eval_output_dir = os.path.join(CONFIG['OUTPUT_DIR'], 'evaluation_results')
print(f"\n📁 Evaluation results will be saved to: {eval_output_dir}")


✓ Loaded config from: /fs/scratch/<allocation>/model_runs/powerline_wavelet_cnn_20251117_211420/config.json

Configuration:
  DATA_FOLDERS: ['/fs/scratch/<allocation>/May29_Alice']
  USE_FOLDERS: ['May29_Alice']
  OUTPUT_BASE_DIR: /fs/scratch/<allocation>/model_runs
  MODEL_NAME: powerline_wavelet_cnn
  ORIGINAL_SAMPLE_RATE: 200000
  SAMPLE_RATE: 200000
  CHUNK_DURATION: 0.1
  CARRIER_FREQ: 20000
  BANDWIDTH: 10000
  DEVICE: cuda
  BATCH_SIZE: 5
  NUM_EPOCHS: 5
  LEARNING_RATE: 0.0001
  TRAIN_SPLIT: 0.8
  USE_GPU: True
  SAVE_EVERY: 5
  EARLY_STOPPING_PATIENCE: 20
  OUTPUT_DIR: /fs/scratch/<allocation>/model_runs/powerline_wavelet_cnn_20251117_211420

📁 Evaluation results will be saved to: /fs/scratch/<allocation>/model_runs/powerline_wavelet_cnn_20251117_211420/evaluation_results


In [4]:
# === HELPER FUNCTIONS ===

def bandpass_filter(signal, low_freq, high_freq, sample_rate, order=4):
    """Apply bandpass filter to signal."""
    nyquist = sample_rate / 2
    low = low_freq / nyquist
    high = high_freq / nyquist
    b, a = butter(order, [low, high], btype='band')
    filtered = filtfilt(b, a, signal)
    return filtered

def lowpass_filter(signal, cutoff_freq, sample_rate, order=4):
    """Apply lowpass filter to signal."""
    nyquist = sample_rate / 2
    cutoff = cutoff_freq / nyquist
    b, a = butter(order, cutoff, btype='low')
    filtered = filtfilt(b, a, signal)
    return filtered

class PowerlineDataset(Dataset):
    """Dataset for 1D powerline signal to USB signal mapping."""
    def __init__(self, chunks):
        self.chunks = chunks
    
    def __len__(self):
        return len(self.chunks)
    
    def __getitem__(self, idx):
        chunk = self.chunks[idx]
        powerline = torch.FloatTensor(chunk['powerline_signal']).unsqueeze(0)
        usb = torch.FloatTensor(chunk['usb_signal'])
        return powerline, usb

print("✓ Helper functions defined")

✓ Helper functions defined


In [5]:
# === MODEL ARCHITECTURE ===
# Import the wavelet-based attention model from the training script

from wavelet_attention import SpectralWaveletAttentionCNN

## Load Trained Model

Load the best model checkpoint from the v4 training run with wavelet attention.

In [6]:
# Load the best trained model
best_model_path = os.path.join(CONFIG['OUTPUT_DIR'], 'best_model.pt')

print(f"Loading trained model from: {best_model_path}")

if not os.path.exists(best_model_path):
    print(f"❌ Model file not found: {best_model_path}")
    print("Please train the model first or check the path.")
    print(f"\nAvailable checkpoints in {CONFIG['OUTPUT_DIR']}:")
    if os.path.exists(CONFIG['OUTPUT_DIR']):
        for f in sorted(os.listdir(CONFIG['OUTPUT_DIR'])):
            if f.endswith('.pt'):
                print(f"  - {f}")
else:
    # Safe checkpoint loading with CPU fallback
    configured_device = str(CONFIG.get('DEVICE', ''))
    if torch.cuda.is_available() and 'cuda' in configured_device.lower():
        map_location = CONFIG['DEVICE']
    else:
        map_location = 'cpu'
        print("CUDA not available — loading checkpoint to CPU (map_location='cpu').")
    
    checkpoint = torch.load(best_model_path, map_location=map_location)
    
    # Create model with wavelet attention architecture
    # Note: v4 uses 0.1 second chunks (shorter than v3's 1 second)
    input_samples = int(CONFIG['CHUNK_DURATION'] * CONFIG['SAMPLE_RATE'])
    
    loaded_model = SpectralWaveletAttentionCNN(
        sample_rate=CONFIG['SAMPLE_RATE'],
        base_channels=64,
        num_scales=50
    )
    
    loaded_model.load_state_dict(checkpoint['model_state_dict'])
    
    # Use the same device as map_location (CPU fallback already determined)
    device = torch.device(map_location)
    loaded_model = loaded_model.to(device)
    loaded_model.eval()
    
    # Update CONFIG['DEVICE'] to reflect actual device being used
    CONFIG['DEVICE'] = str(device)
    
    print(f"✓ Model loaded successfully!")
    print(f"  Epoch: {checkpoint['epoch']}")
    print(f"  Validation Loss: {checkpoint['val_loss']:.6f}")
    print(f"  Training Loss: {checkpoint['train_loss']:.6f}")
    print(f"  Device: {device}")
    print(f"  Input chunk size: {input_samples:,} samples ({CONFIG['CHUNK_DURATION']}s)")
    
    # Count parameters
    total_params = sum(p.numel() for p in loaded_model.parameters())
    trainable_params = sum(p.numel() for p in loaded_model.parameters() if p.requires_grad)
    print(f"\nModel Statistics:")
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    print(f"  Model size: {total_params * 4 / 1024**2:.1f} MB (float32)")

Loading trained model from: /fs/scratch/<allocation>/model_runs/powerline_wavelet_cnn_20251117_211420/best_model.pt
CUDA not available — loading checkpoint to CPU (map_location='cpu').
✓ Model loaded successfully!
  Epoch: 4
  Validation Loss: 0.054313
  Training Loss: 0.027493
  Device: cpu
  Input chunk size: 20,000 samples (0.1s)

Model Statistics:
  Total parameters: 68,762,694
  Trainable parameters: 68,762,694
  Model size: 262.3 MB (float32)
✓ Model loaded successfully!
  Epoch: 4
  Validation Loss: 0.054313
  Training Loss: 0.027493
  Device: cpu
  Input chunk size: 20,000 samples (0.1s)

Model Statistics:
  Total parameters: 68,762,694
  Trainable parameters: 68,762,694
  Model size: 262.3 MB (float32)


In [7]:
# Function to create spectrogram (0-20 kHz)
def create_spectrogram_0_20khz(signal, sample_rate, nperseg=2048, noverlap=1024):
    """Create spectrogram and focus on 0-20 kHz range."""
    f, t, Sxx = spectrogram(signal, fs=sample_rate, nperseg=nperseg, noverlap=noverlap)
    Sxx_db = 10 * np.log10(Sxx + 1e-12)
    freq_mask = (f >= 0) & (f <= 60000)
    f_focused = f[freq_mask]
    Sxx_focused = Sxx_db[freq_mask, :]
    return f_focused, t, Sxx_focused

print("✓ Spectrogram function defined (0-20 kHz range)")

✓ Spectrogram function defined (0-20 kHz range)


## Consecutive Powerline Input Test

Test the model on consecutive powerline input from a full chapter (not random chunks).
Generate predictions, spectrograms, and audio for continuous signal processing.

**Note:** V4 uses shorter 0.1s chunks compared to V3's 1s chunks, which may affect the processing approach.

In [ ]:
# === SELECT CHAPTER AND DURATION FOR CONSECUTIVE TEST ===

CONSECUTIVE_TEST_CHAPTER_INDEX = 0  # Index within validation set
CONSECUTIVE_TEST_DURATION = 10  # Duration in seconds

print("📂 Selecting Chapter from Validation/Test Set")
print("=" * 60)

# Get all available files
data_folder = CONFIG['DATA_FOLDERS'][0] if isinstance(CONFIG['DATA_FOLDERS'], list) else CONFIG['DATA_FOLDERS']
all_test_files = sorted(glob.glob(os.path.join(data_folder, '*_real.bin')))

# Calculate validation files (last 20%)
num_total_files = len(all_test_files)
num_train_files = int(num_total_files * CONFIG['TRAIN_SPLIT'])
validation_files = all_test_files[num_train_files:]

print(f"Total files: {num_total_files}")
print(f"Training files: {num_train_files}")
print(f"Validation files: {len(validation_files)}")
print()

if CONSECUTIVE_TEST_CHAPTER_INDEX >= len(validation_files):
    print(f"❌ Chapter index {CONSECUTIVE_TEST_CHAPTER_INDEX} out of range.")
    print(f"   Available validation chapters: {len(validation_files)}")
else:
    consecutive_powerline_path = validation_files[CONSECUTIVE_TEST_CHAPTER_INDEX]
    consecutive_usb_path = consecutive_powerline_path.replace('_real.bin', '_img.bin')
    consecutive_chapter_name = os.path.basename(consecutive_powerline_path).replace('_real.bin', '')
    
    print(f"✓ Selected from Validation Set:")
    print(f"  Chapter: {consecutive_chapter_name}")
    print(f"  Index in validation set: {CONSECUTIVE_TEST_CHAPTER_INDEX}")
    print(f"  Duration: {CONSECUTIVE_TEST_DURATION} seconds")
    print("=" * 60)

📂 Selecting Chapter from Validation/Test Set
Total files: 12
Training files: 9
Validation files: 3

✓ Selected from Validation Set:
  Chapter: Chap_7
  Index in validation set: 0
  Duration: 30 seconds


In [9]:
# === LOAD AND PROCESS CONSECUTIVE POWERLINE INPUT ===

print("\n🔄 Loading Consecutive Powerline Data")
print("=" * 60)

num_samples_consecutive = CONSECUTIVE_TEST_DURATION * CONFIG['SAMPLE_RATE']
consecutive_powerline_raw = np.fromfile(consecutive_powerline_path, dtype=np.float32, count=num_samples_consecutive)
consecutive_usb_raw = np.fromfile(consecutive_usb_path, dtype=np.float32, count=num_samples_consecutive)

print(f"Loaded {len(consecutive_powerline_raw):,} samples")
print(f"Duration: {len(consecutive_powerline_raw) / CONFIG['SAMPLE_RATE']:.2f} seconds")

# Use raw downsampled powerline (no pre-bandpass)
print("\n🔧 Using raw downsampled powerline (no bandpass)")
consecutive_powerline_normalized = consecutive_powerline_raw / (np.max(np.abs(consecutive_powerline_raw)) + 1e-8)

print(f"\n✓ Data prepared for inference")
print("=" * 60)


🔄 Loading Consecutive Powerline Data
Loaded 6,000,000 samples
Duration: 30.00 seconds

🔧 Using raw downsampled powerline (no bandpass)

✓ Data prepared for inference


In [10]:
# === RUN INFERENCE ON CONSECUTIVE INPUT (chunked) ===

print("\n🤖 Running Model Inference on Consecutive Input (chunked)")
print("=" * 60)

chunk_len = input_samples
total_len = len(consecutive_powerline_normalized)
num_chunks = int(np.ceil(total_len / chunk_len))

print(f"Total samples: {total_len:,}, Chunk length: {chunk_len:,}, Num chunks: {num_chunks}")
print(f"Note: V4 uses {CONFIG['CHUNK_DURATION']}s chunks ({chunk_len:,} samples)")

pred_chunks = []
loaded_model.eval()
with torch.no_grad():
    for ci in tqdm(range(num_chunks), desc="Processing chunks"):
        s = ci * chunk_len
        e = min((ci + 1) * chunk_len, total_len)
        seg = consecutive_powerline_normalized[s:e]

        if len(seg) < chunk_len:
            pad = np.zeros(chunk_len, dtype=np.float32)
            pad[:len(seg)] = seg
            seg_input = pad
            need_unpad = True
            unpad_len = len(seg)
        else:
            seg_input = seg
            need_unpad = False

        seg_tensor = torch.FloatTensor(seg_input).unsqueeze(0).unsqueeze(0).to(CONFIG['DEVICE'])
        out = loaded_model(seg_tensor).cpu().squeeze().numpy()

        if need_unpad:
            out = out[:unpad_len]

        pred_chunks.append(out)

consecutive_usb_pred = np.concatenate(pred_chunks, axis=0)
consecutive_usb_pred = consecutive_usb_pred[:total_len]

print(f"\n✓ Inference complete")
print(f"Output shape: {consecutive_usb_pred.shape}")
print(f"Output samples: {len(consecutive_usb_pred):,}")
print(f"Output duration: {len(consecutive_usb_pred) / CONFIG['SAMPLE_RATE']:.2f} seconds")

true_usb_for_metrics = consecutive_usb_raw[:len(consecutive_usb_pred)]
consecutive_correlation = np.corrcoef(consecutive_usb_pred, true_usb_for_metrics)[0, 1]
consecutive_mse = np.mean((consecutive_usb_pred - true_usb_for_metrics) ** 2)
consecutive_mae = np.mean(np.abs(consecutive_usb_pred - true_usb_for_metrics))

print(f"\n📊 Performance Metrics:")
print(f"  Correlation: {consecutive_correlation:.4f}")
print(f"  MSE: {consecutive_mse:.6f}")
print(f"  MAE: {consecutive_mae:.6f}")
print("=" * 60)


🤖 Running Model Inference on Consecutive Input (chunked)
Total samples: 6,000,000, Chunk length: 20,000, Num chunks: 300
Note: V4 uses 0.1s chunks (20,000 samples)


Processing chunks:   0%|          | 1/300 [02:59<14:52:38, 179.13s/it]



KeyboardInterrupt: 

In [ ]:
# === VISUALIZE TIME-DOMAIN COMPARISON ===

print("\n📈 Visualizing Time-Domain Signals")
print("=" * 60)

time_axis_full = np.arange(len(consecutive_usb_pred)) / CONFIG['SAMPLE_RATE']

fig, axes = plt.subplots(3, 1, figsize=(18, 10))

axes[0].plot(time_axis_full, consecutive_usb_raw[:len(consecutive_usb_pred)], 
             color='red', linewidth=0.5, alpha=0.8, label='True USB')
axes[0].set_title(f'True USB Signal - {consecutive_chapter_name}', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Amplitude', fontsize=12)
axes[0].legend(loc='upper right', fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, CONSECUTIVE_TEST_DURATION)

axes[1].plot(time_axis_full, consecutive_usb_pred, 
             color='blue', linewidth=0.5, alpha=0.8, label='Predicted USB')
axes[1].set_title(f'Predicted USB Signal (Correlation: {consecutive_correlation:.4f})', 
                  fontsize=14, fontweight='bold')
axes[1].set_ylabel('Amplitude', fontsize=12)
axes[1].legend(loc='upper right', fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, CONSECUTIVE_TEST_DURATION)

axes[2].plot(time_axis_full, consecutive_powerline_normalized[:len(consecutive_usb_pred)], 
             color='orange', linewidth=0.5, alpha=0.8, label='Powerline Input (Normalized)')
axes[2].set_title(f'Powerline Input Signal', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Time (seconds)', fontsize=12)
axes[2].set_ylabel('Amplitude', fontsize=12)
axes[2].legend(loc='upper right', fontsize=10)
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim(0, CONSECUTIVE_TEST_DURATION)

plt.tight_layout()

# Save figure
time_domain_path = os.path.join(eval_output_dir, f'time_domain_{consecutive_chapter_name}.png')
plt.savefig(time_domain_path, dpi=150, bbox_inches='tight')

print(f"✓ Saved time-domain plot to: {time_domain_path}")print("=" * 60)

print("✓ Time-domain visualization complete")

plt.show()

In [ ]:
# === SPECTROGRAMS FOR TRUE vs PREDICTED (consecutive) ===

print("\n📊 Computing Spectrograms (0-20 kHz)")
print("=" * 60)

nperseg_consec = 2048
noverlap_consec = nperseg_consec // 2

f_true_consec, t_true_consec, Sxx_true_consec = signal.spectrogram(
    consecutive_usb_raw[:len(consecutive_usb_pred)],
    fs=CONFIG['SAMPLE_RATE'],
    nperseg=nperseg_consec,
    noverlap=noverlap_consec
)

f_pred_consec, t_pred_consec, Sxx_pred_consec = signal.spectrogram(
    consecutive_usb_pred,
    fs=CONFIG['SAMPLE_RATE'],
    nperseg=nperseg_consec,
    noverlap=noverlap_consec
)

freq_limit = CONFIG.get('CARRIER_FREQ', CONFIG.get('SAMPLE_RATE')/2)
mask_true = f_true_consec <= freq_limit
mask_pred = f_pred_consec <= freq_limit

f_true_plot = f_true_consec[mask_true]
Sxx_true_plot = Sxx_true_consec[mask_true, :]
f_pred_plot = f_pred_consec[mask_pred]
Sxx_pred_plot = Sxx_pred_consec[mask_pred, :]

spec_true_db = 10 * np.log10(Sxx_true_plot + 1e-12)
spec_pred_db = 10 * np.log10(Sxx_pred_plot + 1e-12)
spec_diff = spec_pred_db - spec_true_db

print(f"Spectrogram frequency range: 0 - {freq_limit/1000:.0f} kHz")
print(f"Time range: 0 - {t_true_consec[-1]:.2f} seconds")

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

im1 = axes[0].pcolormesh(t_true_consec, f_true_plot/1000, spec_true_db, shading='gouraud', cmap='viridis')
axes[0].set_title(f'True USB Spectrogram - {consecutive_chapter_name}', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Frequency (kHz)', fontsize=12)
fig.colorbar(im1, ax=axes[0], label='Power (dB)')

im2 = axes[1].pcolormesh(t_pred_consec, f_pred_plot/1000, spec_pred_db, shading='gouraud', cmap='viridis')
axes[1].set_title('Predicted USB Spectrogram', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Frequency (kHz)', fontsize=12)
fig.colorbar(im2, ax=axes[1], label='Power (dB)')

im3 = axes[2].pcolormesh(t_pred_consec, f_pred_plot/1000, spec_diff, shading='gouraud', cmap='RdBu_r')
axes[2].set_title('Difference (Predicted - True)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Time (seconds)', fontsize=12)
axes[2].set_ylabel('Frequency (kHz)', fontsize=12)
fig.colorbar(im3, ax=axes[2], label='Power Difference (dB)')

plt.tight_layout()

# Save figure
spectrogram_path = os.path.join(eval_output_dir, f'spectrogram_{consecutive_chapter_name}.png')
plt.savefig(spectrogram_path, dpi=150, bbox_inches='tight')

print(f"✓ Saved spectrogram plot to: {spectrogram_path}")print("=" * 60)

print("\n✓ Spectrogram visualization complete")

plt.show()

In [ ]:
# === AUDIO PLAYBACK FOR CONSECUTIVE TEST ===

print("\n🎧 Audio Playback - Consecutive Input Test")
print("=" * 60)

playback_duration_consec = min(30.0, len(consecutive_usb_pred) / CONFIG['SAMPLE_RATE'])
playback_samples_consec = int(playback_duration_consec * CONFIG['SAMPLE_RATE'])

print(f"Playback Duration: {playback_duration_consec:.1f} seconds")
print(f"Playback Samples: {playback_samples_consec:,}")
print(f"\nChapter: {consecutive_chapter_name}")
print(f"Model: Wavelet Attention CNN (V4)")
print(f"Correlation: {consecutive_correlation:.4f}")
print(f"MSE: {consecutive_mse:.6f}")
print(f"MAE: {consecutive_mae:.6f}")

print("\n🔊 True USB Signal:")
display(Audio(consecutive_usb_raw[:playback_samples_consec], rate=CONFIG['SAMPLE_RATE']))

print("\n🔊 Predicted USB Signal:")
display(Audio(consecutive_usb_pred[:playback_samples_consec], rate=CONFIG['SAMPLE_RATE']))

print("\n🔊 Powerline Input (Raw):")
display(Audio(consecutive_powerline_normalized[:playback_samples_consec], rate=CONFIG['SAMPLE_RATE']))

print("\n" + "=" * 60)
print("✓ Consecutive input test complete!")

# Save performance metrics and results summary
results_summary = {
    'chapter': consecutive_chapter_name,
    'duration_seconds': CONSECUTIVE_TEST_DURATION,
    'model_type': 'Wavelet Attention CNN (V4)',
    'chunk_duration': CONFIG['CHUNK_DURATION'],
    'chunk_samples': input_samples,
    'total_output_samples': int(len(consecutive_usb_pred)),
    'correlation': float(consecutive_correlation),
    'mse': float(consecutive_mse),
    'mae': float(consecutive_mae),
    'sample_rate': CONFIG['SAMPLE_RATE'],
    'model_parameters': total_params,
    'device_used': str(device)
}

results_json_path = os.path.join(eval_output_dir, f'results_{consecutive_chapter_name}.json')
with open(results_json_path, 'w') as f:
    json.dump(results_summary, f, indent=2)
print(f"✓ Saved results summary to: {results_json_path}")

# Save predicted audio
pred_audio_path = os.path.join(eval_output_dir, f'predicted_usb_{consecutive_chapter_name}.npy')
np.save(pred_audio_path, consecutive_usb_pred)
print(f"✓ Saved predicted audio to: {pred_audio_path}")

print(f"\nSummary:")
print(f"  - Chapter: {consecutive_chapter_name}")
print(f"  - Duration: {CONSECUTIVE_TEST_DURATION} seconds")
print(f"  - Model: Wavelet Attention CNN (V4)")
print(f"  - Chunk size: {CONFIG['CHUNK_DURATION']}s ({input_samples:,} samples)")
print(f"  - Input: Consecutive powerline signal")
print(f"  - Output: {len(consecutive_usb_pred):,} samples")
print(f"  - Correlation: {consecutive_correlation:.4f}")
print(f"  - MSE: {consecutive_mse:.6f}")
print(f"  - MAE: {consecutive_mae:.6f}")
print(f"\n📁 All results saved to: {eval_output_dir}")
print("=" * 60)


## Load and Display Saved Results

Load previously saved evaluation results from the evaluation_results directory.

In [13]:
# === LOAD SAVED EVALUATION RESULTS ===

print("📂 Loading Saved Evaluation Results")
print("=" * 60)

# Check if evaluation results directory exists
if not os.path.exists(eval_output_dir):
    print(f"❌ Evaluation results directory not found: {eval_output_dir}")
    print("Please run the evaluation first.")
else:
    # List all available result files
    result_files = sorted([f for f in os.listdir(eval_output_dir) if f.endswith('.json')])
    
    if not result_files:
        print("❌ No result files found in evaluation directory.")
    else:
        print(f"✓ Found {len(result_files)} result file(s):")
        for rf in result_files:
            print(f"  - {rf}")
        
        # Load all JSON results
        print(f"\n📊 Loading Results:")
        print("=" * 60)
        
        all_results = []
        for result_file in result_files:
            result_path = os.path.join(eval_output_dir, result_file)
            with open(result_path, 'r') as f:
                result_data = json.load(f)
            all_results.append(result_data)
            
            print(f"\n{result_file}:")
            print(f"  Chapter: {result_data['chapter']}")
            print(f"  Model: {result_data['model_type']}")
            print(f"  Duration: {result_data['duration_seconds']}s")
            print(f"  Chunk Size: {result_data['chunk_duration']}s ({result_data['chunk_samples']:,} samples)")
            print(f"  Output Samples: {result_data['total_output_samples']:,}")
            print(f"  Correlation: {result_data['correlation']:.4f}")
            print(f"  MSE: {result_data['mse']:.6f}")
            print(f"  MAE: {result_data['mae']:.6f}")
            print(f"  Sample Rate: {result_data['sample_rate']:,} Hz")
            print(f"  Model Parameters: {result_data['model_parameters']:,}")
            print(f"  Device: {result_data['device_used']}")
        
        # Display summary table if multiple results
        if len(all_results) > 1:
            print(f"\n📋 Summary Table:")
            print("=" * 60)
            print(f"{'Chapter':<20} {'Correlation':<12} {'MSE':<12} {'MAE':<12}")
            print("-" * 60)
            for r in all_results:
                print(f"{r['chapter']:<20} {r['correlation']:<12.4f} {r['mse']:<12.6f} {r['mae']:<12.6f}")
            
            # Calculate average metrics
            avg_corr = np.mean([r['correlation'] for r in all_results])
            avg_mse = np.mean([r['mse'] for r in all_results])
            avg_mae = np.mean([r['mae'] for r in all_results])
            
            print("-" * 60)
            print(f"{'AVERAGE':<20} {avg_corr:<12.4f} {avg_mse:<12.6f} {avg_mae:<12.6f}")
            print("=" * 60)
        
        # List available image files
        print(f"\n🖼️  Available Visualization Files:")
        print("=" * 60)
        image_files = sorted([f for f in os.listdir(eval_output_dir) if f.endswith('.png')])
        if image_files:
            for img_file in image_files:
                img_path = os.path.join(eval_output_dir, img_file)
                file_size = os.path.getsize(img_path) / 1024  # KB
                print(f"  - {img_file} ({file_size:.1f} KB)")
        else:
            print("  No image files found.")
        
        # List available audio files
        print(f"\n🔊 Available Predicted Audio Files:")
        print("=" * 60)
        audio_files = sorted([f for f in os.listdir(eval_output_dir) if f.endswith('.npy')])
        if audio_files:
            for audio_file in audio_files:
                audio_path = os.path.join(eval_output_dir, audio_file)
                file_size = os.path.getsize(audio_path) / 1024  # KB
                print(f"  - {audio_file} ({file_size:.1f} KB)")
        else:
            print("  No audio files found.")
        
        print(f"\n📁 All files located in: {eval_output_dir}")
        print("=" * 60)

📂 Loading Saved Evaluation Results
✓ Found 1 result file(s):
  - results_Chap_7.json

📊 Loading Results:

results_Chap_7.json:
  Chapter: Chap_7
  Model: Wavelet Attention CNN (V4)
  Duration: 10s
  Chunk Size: 0.1s (20,000 samples)
  Output Samples: 2,000,000
  Correlation: 0.5614
  MSE: 0.039397
  MAE: 0.166174
  Sample Rate: 200,000 Hz
  Model Parameters: 68,762,694
  Device: cuda

🖼️  Available Visualization Files:
  - spectrogram_Chap_7.png (4320.8 KB)
  - time_domain_Chap_7.png (420.5 KB)

🔊 Available Predicted Audio Files:
  - predicted_usb_Chap_7.npy (7812.6 KB)

📁 All files located in: /fs/scratch/<allocation>/model_runs/powerline_wavelet_cnn_20251117_211420/evaluation_results


## Load and Play Saved Predicted Audio

Load a saved predicted audio file and play it.

In [14]:
# === LOAD AND PLAY SAVED PREDICTED AUDIO ===

# Select which audio file to load (change index as needed)
AUDIO_FILE_INDEX = 0

audio_files = sorted([f for f in os.listdir(eval_output_dir) if f.endswith('.npy')])

if audio_files and AUDIO_FILE_INDEX < len(audio_files):
    selected_audio_file = audio_files[AUDIO_FILE_INDEX]
    audio_file_path = os.path.join(eval_output_dir, selected_audio_file)
    
    print(f"🔊 Loading Saved Audio: {selected_audio_file}")
    print("=" * 60)
    
    # Load the predicted audio
    loaded_pred_audio = np.load(audio_file_path)
    
    print(f"Audio shape: {loaded_pred_audio.shape}")
    print(f"Duration: {len(loaded_pred_audio) / CONFIG['SAMPLE_RATE']:.2f} seconds")
    print(f"Sample rate: {CONFIG['SAMPLE_RATE']:,} Hz")
    
    # Play audio (first 30 seconds)
    playback_duration = min(30.0, len(loaded_pred_audio) / CONFIG['SAMPLE_RATE'])
    playback_samples = int(playback_duration * CONFIG['SAMPLE_RATE'])
    
    print(f"\n🔊 Playing {playback_duration:.1f} seconds:")
    display(Audio(loaded_pred_audio[:playback_samples], rate=CONFIG['SAMPLE_RATE']))
    
    print("=" * 60)
else:
    print("❌ No audio files found or invalid index.")

🔊 Loading Saved Audio: predicted_usb_Chap_7.npy
Audio shape: (2000000,)
Duration: 10.00 seconds
Sample rate: 200,000 Hz

🔊 Playing 10.0 seconds:
